# Patent Phrase Matching â€” Complete Assignment Notebook

This notebook implements the required workflow for the **U.S. Patent Phrase to Phrase Matching** assignment, using the uploaded reference notebook to align the section order and deliverables.

**Included tasks**
1. Fine-tuned DeBERTa cross-encoder baseline.
2. Frozen pretrained encoder cosine-similarity baseline (no task fine-tuning).
3. Two controlled experiments on the same anchor-grouped validation split.
4. Validation error analysis and per-CPC-section metrics.
5. Retrain the best validation configuration on all labeled data and create `submission.csv`.

**Kaggle setup**
- Attach the competition data through **Add Input â†’ Competitions**.
- Attach a local DeBERTa checkpoint as a Kaggle Dataset for offline submission runs.
- Select a GPU accelerator (T4 or P100, if available).
- Do not reinstall PyTorch unless diagnostics show the preinstalled package is broken; Kaggle's environment is preconfigured.
- Submitted code competitions run with internet disabled, so a Hub download is only for initial setup, not the final offline run.

**Important:** this notebook has been syntax-checked, but its full training run and leaderboard submission have not been executed here. Do not report validation or leaderboard scores until your Kaggle run produces them.


## 1. Setup and reproducibility

In [1]:
# Environment diagnostics (does not import torch)
# Run this if the next setup cell fails at `import torch`.
import sys
import importlib.metadata as metadata

print("Python:", sys.version)
for package in ["torch", "torchvision", "numpy", "transformers", "datasets", "tokenizers"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\nIf `import torch` fails with a SystemError:")
print("1. Restart the Kaggle session and run the notebook from the first cell.")
print("2. Remove any earlier `pip install/upgrade torch` commands from the notebook.")
print("3. If it still fails, use a fresh Kaggle session/runtime and inspect these package versions.")
print("Do not continue to model training until `import torch` succeeds.")


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
torch: 2.11.0+cu128
torchvision: 0.26.0+cu128
numpy: 2.1.3
transformers: 5.16.1
datasets: 4.8.5
tokenizers: 0.23.1

If `import torch` fails with a SystemError:
1. Restart the Kaggle session and run the notebook from the first cell.
2. Remove any earlier `pip install/upgrade torch` commands from the notebook.
3. If it still fails, use a fresh Kaggle session/runtime and inspect these package versions.
Do not continue to model training until `import torch` succeeds.


In [2]:
import os, glob, time, json, shutil, warnings, random, inspect
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics.pairwise import cosine_similarity
from tokenizers.processors import TemplateProcessing
from transformers import (AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
transformers.set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__, "transformers:", transformers.__version__, "device:", device)

torch: 2.11.0+cu128 transformers: 5.16.1 device: cuda


## 2. Find competition data and checkpoint (download is included here)

In [3]:
MODEL_ID = "microsoft/deberta-v3-small"
MODEL_NAME = MODEL_ID.split("/")[-1]
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def find_file(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None

def find_model_dir():
    # Prefer a local checkpoint attached as a Kaggle input.
    hits = glob.glob(f"/kaggle/input/**/{MODEL_NAME}/config.json", recursive=True)
    if hits:
        return str(Path(hits[0]).parent)
    hits = glob.glob(f"/kaggle/input/**/config.json", recursive=True)
    for p in hits:
        try:
            import json
            cfg=json.load(open(p))
            if "deberta" in str(cfg.get("model_type", "")).lower():
                return str(Path(p).parent)
        except Exception:
            pass
    return None

MODEL_PATH = find_model_dir()
if MODEL_PATH is None:
    # This succeeds only when Internet is enabled. In an offline competition run,
    # attach the saved checkpoint as a Kaggle input first.
    try:
        from huggingface_hub import snapshot_download
        MODEL_PATH = snapshot_download(
            repo_id=MODEL_ID,
            local_dir=str(OUT_DIR / MODEL_NAME),
            allow_patterns=["*.json", "*.safetensors", "*.bin", "*.model", "*.txt", "*.vocab"]
        )
    except Exception as e:
        raise RuntimeError(
            "DeBERTa checkpoint not found locally and download failed. Enable Internet for the initial download, "
            "then save the checkpoint as a Kaggle Dataset and attach it as an input for offline submission."
        ) from e
print("Model checkpoint:", MODEL_PATH)

train_path = find_file("train.csv")
test_path = find_file("test.csv")
assert train_path and test_path, "Competition train.csv/test.csv not found. Add the competition data as an input."
train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)
required_train={"id","anchor","target","context","score"}
required_test={"id","anchor","target","context"}
assert required_train.issubset(train_raw.columns), train_raw.columns
assert required_test.issubset(test_raw.columns), test_raw.columns
print("Train:", train_raw.shape, "Test:", test_raw.shape)
print("Training score distribution:\n", train_raw.score.value_counts().sort_index())

Model checkpoint: /kaggle/input/notebooks/charankailasa/patent-weights/deberta-v3-small
Train: (36473, 5) Test: (36, 4)
Training score distribution:
 score
0.00     7471
0.25    11519
0.50    12300
0.75     4029
1.00     1154
Name: count, dtype: int64


## 3. Shared validation split and context features

In [4]:
CPC_SECTIONS = {
    "A": "human necessities", "B": "performing operations; transporting",
    "C": "chemistry; metallurgy", "D": "textiles; paper", "E": "fixed constructions",
    "F": "mechanical engineering; lighting; heating; weapons; blasting",
    "G": "physics", "H": "electricity",
}

def prepare_frame(df, context_mode="section"):
    out=df.copy()
    out["anchor"] = out["anchor"].fillna("").astype(str)
    out["target"] = out["target"].fillna("").astype(str)
    out["context"] = out["context"].fillna("").astype(str)
    if context_mode == "section":
        ctx=out["context"].str[0].map(CPC_SECTIONS).fillna("unknown technical field")
    elif context_mode == "raw_code":
        ctx=out["context"]
    elif context_mode == "none":
        ctx=pd.Series([""]*len(out), index=out.index)
    else:
        raise ValueError(context_mode)
    out["text_a"] = out["anchor"]
    out["text_b"] = (out["target"] + ("; " + ctx if context_mode != "none" else "")).str.strip()
    return out

SEED=42
VAL_FRACTION=0.2
splitter=GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
tr_idx, va_idx=next(splitter.split(train_raw, groups=train_raw["anchor"].fillna("")))
train_base=train_raw.iloc[tr_idx].reset_index(drop=True)
val_base=train_raw.iloc[va_idx].reset_index(drop=True)
assert set(train_base.anchor).isdisjoint(set(val_base.anchor)), "Anchor leakage detected"
print(f"Train rows={len(train_base):,}; validation rows={len(val_base):,}; held-out anchors={val_base.anchor.nunique():,}")

Train rows=28,914; validation rows=7,559; held-out anchors=147


## 4. Tokenizer, datasets, and training utilities

The same held-out-anchor split is used for all experiments. Each experiment changes one factor from the baseline.

In [5]:
MAX_LEN=64
BATCH_SIZE=32

tokenizer=AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
# Ensure paired inputs have the standard [CLS] A [SEP] B [SEP] template.
try:
    tokenizer.backend_tokenizer.post_processor = TemplateProcessing(
        single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B:1 [SEP]:1",
        special_tokens=[("[CLS]", tokenizer.cls_token_id), ("[SEP]", tokenizer.sep_token_id)]
    )
except Exception as e:
    print("Tokenizer template left unchanged:", repr(e))

def make_dataset(df, labeled=True, context_mode="section"):
    frame=prepare_frame(df, context_mode=context_mode)
    keep=["text_a","text_b"] + (["score"] if labeled else [])
    frame=frame[keep].copy()
    if labeled: frame["score"]=frame["score"].astype("float32")
    ds=Dataset.from_pandas(frame, preserve_index=False)
    ds=ds.map(lambda b: tokenizer(b["text_a"], b["text_b"], truncation=True, max_length=MAX_LEN),
               batched=True, remove_columns=["text_a","text_b"])
    if labeled: ds=ds.rename_column("score","labels")
    return ds

def pearson(y_true, y_pred):
    y_true=np.asarray(y_true).reshape(-1); y_pred=np.asarray(y_pred).reshape(-1)
    if len(y_true)<2 or np.std(y_true)==0 or np.std(y_pred)==0: return float("nan")
    return float(np.corrcoef(y_true,y_pred)[0,1])

def compatible_training_arguments(**kwargs):
    """Build TrainingArguments across Kaggle Transformers versions."""
    params = inspect.signature(TrainingArguments.__init__).parameters
    kwargs = dict(kwargs)

    # Transformers versions differ on this argument name.
    if "eval_strategy" in kwargs and "eval_strategy" not in params:
        value = kwargs.pop("eval_strategy")
        if "evaluation_strategy" in params:
            kwargs["evaluation_strategy"] = value
    if "evaluation_strategy" in kwargs and "evaluation_strategy" not in params:
        value = kwargs.pop("evaluation_strategy")
        if "eval_strategy" in params:
            kwargs["eval_strategy"] = value

    # Older releases do not support warmup_ratio. Convert it to warmup_steps.
    if "warmup_ratio" in kwargs and "warmup_ratio" not in params:
        ratio = kwargs.pop("warmup_ratio")
        steps = kwargs.get("max_steps", -1)
        if steps is None or steps <= 0:
            # Trainer computes the exact schedule from the dataloader; zero is
            # a safe compatibility fallback if the old API lacks warmup_ratio.
            kwargs.setdefault("warmup_steps", 0)
        else:
            kwargs.setdefault("warmup_steps", int(steps * ratio))

    # Avoid version-specific keyword errors on older Kaggle images.
    kwargs = {key: value for key, value in kwargs.items() if key in params}
    return TrainingArguments(**kwargs)


def train_one_experiment(name, tr_df, va_df, context_mode="section", lr=2e-5, epochs=3, batch_size=BATCH_SIZE):
    start=time.time()
    ds_tr=make_dataset(tr_df, True, context_mode)
    ds_va=make_dataset(va_df, True, context_mode)
    model=AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, num_labels=1, dtype=torch.float32, local_files_only=True
    )
    args=compatible_training_arguments(
        output_dir=f"/kaggle/working/{name}_trainer",
        num_train_epochs=epochs,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size*2,
        warmup_ratio=0.1, weight_decay=0.01,
        eval_strategy="epoch", save_strategy="no",
        logging_steps=100, fp16=torch.cuda.is_available(),
        report_to="none", seed=SEED, data_seed=SEED,
        dataloader_num_workers=2,
    )
    trainer=Trainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
                    data_collator=DataCollatorWithPadding(tokenizer))
    trainer.train()
    pred=trainer.predict(ds_va).predictions.reshape(-1)
    score=pearson(va_df["score"].to_numpy(), pred)
    elapsed=time.time()-start
    out=va_df.copy(); out["pred"]=pred
    print(f"{name}: validation Pearson={score:.5f}; training+validation inference={elapsed/60:.1f} min")
    # Release GPU memory before the next experiment.
    del trainer, model, ds_tr, ds_va
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return {"name":name,"pearson":score,"minutes":elapsed/60,"predictions":out,"context_mode":context_mode,"lr":lr,"epochs":epochs,"batch_size":batch_size}

[transformers] The tokenizer you are loading from '/kaggle/input/notebooks/charankailasa/patent-weights/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


## 5. Task 1 â€” fine-tuned baseline

In [6]:
baseline=train_one_experiment("task1_baseline", train_base, val_base, context_mode="section", lr=2e-5, epochs=3)
print("Task 1 baseline validation Pearson:", round(baseline["pearson"], 5))
print("Anchor-grouped split rationale: complete anchors are held out so validation measures generalization to unseen anchors. A random row split could place the same anchor in both partitions and inflate the score through anchor-specific familiarity.")

Map:   0%|          | 0/28914 [00:00<?, ? examples/s]

Map:   0%|          | 0/7559 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/input/notebooks/charankailasa/patent-weights/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight   

Epoch,Training Loss,Validation Loss
1,0.062111,0.051993
2,0.049710,0.050423
3,0.040349,0.050927


task1_baseline: validation Pearson=0.81601; training+validation inference=5.9 min
Task 1 baseline validation Pearson: 0.81601
Anchor-grouped split rationale: complete anchors are held out so validation measures generalization to unseen anchors. A random row split could place the same anchor in both partitions and inflate the score through anchor-specific familiarity.


## 6. Task 2 â€” pretrained model without fine-tuning (bi-encoder-style cosine baseline)

In [7]:
# Use the pretrained encoder as a frozen phrase encoder; each phrase is encoded separately.
# This is a lightweight cosine-similarity baseline using the same attached DeBERTa checkpoint.
# It is not a separately trained Sentence-Transformers checkpoint.
@torch.no_grad()
def encode_phrases(texts, batch_size=64):
    encoder=AutoModel.from_pretrained(MODEL_PATH, local_files_only=True).to(device)
    encoder.eval()
    vectors=[]
    for i in range(0, len(texts), batch_size):
        toks=tokenizer(list(texts[i:i+batch_size]), padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
        toks={k:v.to(device) for k,v in toks.items()}
        out=encoder(**toks)
        hidden=out.last_hidden_state
        mask=toks["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled=(hidden*mask).sum(1)/mask.sum(1).clamp(min=1)
        vectors.append(torch.nn.functional.normalize(pooled, p=2, dim=1).cpu().numpy())
    del encoder
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.concatenate(vectors, axis=0)

val_embed=prepare_frame(val_base, "section")
unique_phrases=pd.Index(pd.concat([val_embed["anchor"], val_embed["target"]], ignore_index=True).unique())
phrase_vec=encode_phrases(unique_phrases.to_numpy())
vec_map={p:phrase_vec[i] for i,p in enumerate(unique_phrases)}
bi_pred=np.array([float(np.dot(vec_map[a],vec_map[t])) for a,t in zip(val_embed.anchor,val_embed.target)])
# Cosine similarity naturally ranges [-1,1]; map to [0,1] for a readable prediction scale.
bi_pred=np.clip((bi_pred+1.0)/2.0, 0, 1)
bi_pearson=pearson(val_base.score, bi_pred)
print(f"Frozen bi-encoder-style cosine baseline Pearson: {bi_pearson:.5f}")
print("Cross-encoder vs bi-encoder: the cross-encoder jointly attends across both phrases and can learn token-level interactions during fine-tuning. The frozen bi-encoder encodes each phrase independently and compares pooled vectors with cosine similarity; it receives no task-specific training.")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: /kaggle/input/notebooks/charankailasa/patent-weights/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical 

Frozen bi-encoder-style cosine baseline Pearson: 0.19791
Cross-encoder vs bi-encoder: the cross-encoder jointly attends across both phrases and can learn token-level interactions during fine-tuning. The frozen bi-encoder encodes each phrase independently and compares pooled vectors with cosine similarity; it receives no task-specific training.


## 7. Task 3 â€” two controlled experiments

In [8]:
# Experiment A changes only context representation: section title -> no context.
exp_no_context=train_one_experiment("task3_no_context", train_base, val_base, context_mode="none", lr=2e-5, epochs=3)

# Experiment B changes only learning rate from 2e-5 -> 1e-5; keep section context and 3 epochs.
exp_low_lr=train_one_experiment("task3_lr_1e5", train_base, val_base, context_mode="section", lr=1e-5, epochs=3)

results=pd.DataFrame([
    {"setting":"Fine-tuned baseline: section context, LR=2e-5", "validation_pearson":baseline["pearson"], "minutes":baseline["minutes"], "interpretation":"Reference configuration."},
    {"setting":"Experiment A: no context", "validation_pearson":exp_no_context["pearson"], "minutes":exp_no_context["minutes"], "interpretation":"Measures contribution of CPC section context."},
    {"setting":"Experiment B: LR=1e-5", "validation_pearson":exp_low_lr["pearson"], "minutes":exp_low_lr["minutes"], "interpretation":"Measures sensitivity to a lower learning rate."},
])
results.to_csv("/kaggle/working/experiment_results.csv", index=False)
display(results)
print("Interpret small differences cautiously; a Pearson difference around 0.005 can be random-seed noise. This notebook does not rerun seeds automatically.")

Map:   0%|          | 0/28914 [00:00<?, ? examples/s]

Map:   0%|          | 0/7559 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/input/notebooks/charankailasa/patent-weights/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight   

Epoch,Training Loss,Validation Loss
1,0.061732,0.053063
2,0.049058,0.048379
3,0.041255,0.048872


...

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/input/notebooks/charankailasa/patent-weights/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight   

Epoch,Training Loss,Validation Loss
1,0.066989,0.053762
2,0.056869,0.050436
3,0.048847,0.050171


task3_lr_1e5: validation Pearson=0.81326; training+validation inference=5.8 min


,setting,validation_pearson,minutes,interpretation
0,"Fine-tuned baseline: section context, LR=2e-5",0.816007,5.853882,Reference configuration.
1,Experiment A: no context,0.816755,5.392378,Measures contribution of CPC section context.
2,Experiment B: LR=1e-5,0.813263,5.795645,Measures sensitivity to a lower learning rate.


Interpret small differences cautiously; a Pearson difference around 0.005 can be random-seed noise. This notebook does not rerun seeds automatically.


## 8. Select best validation model and Task 4 â€” error analysis

In [9]:
trained_runs=[baseline, exp_no_context, exp_low_lr]
best=max(trained_runs, key=lambda x: x["pearson"] if np.isfinite(x["pearson"]) else -np.inf)
val_pred=best["predictions"].copy()
val_pred["abs_error"]=(val_pred["score"]-val_pred["pred"]).abs()
val_pred["cpc_section"]=val_pred["context"].astype(str).str[0]
val_pred.to_csv("/kaggle/working/val_predictions.csv", index=False)
print("Best configuration by validation Pearson:", best["name"], best["pearson"])

# Pearson by CPC section A-H
section_rows=[]
for section, grp in val_pred.groupby("cpc_section"):
    section_rows.append({"CPC section":section,"rows":len(grp),"Pearson":pearson(grp.score,grp.pred)})
section_table=pd.DataFrame(section_rows).sort_values("CPC section")
display(section_table)
section_table.to_csv("/kaggle/working/pearson_by_cpc_section.csv", index=False)

# Mean prediction at each ground-truth score; highlights regression-to-the-mean/compression.
score_table=(val_pred.groupby("score",as_index=False).agg(mean_prediction=("pred","mean"), count=("pred","size")))
display(score_table)
score_table.to_csv("/kaggle/working/mean_prediction_by_true_score.csv", index=False)

# Ten largest absolute errors
worst=val_pred.nlargest(10,"abs_error")[["id","anchor","target","context","score","pred","abs_error"]]
display(worst)
worst.to_csv("/kaggle/working/top10_validation_errors.csv", index=False)

Best configuration by validation Pearson: task3_no_context 0.8167549704839546


,CPC section,rows,Pearson
0,A,943,0.785087
1,B,1926,0.892108
2,C,1177,0.771534
3,D,199,0.786492
4,E,373,0.806240
5,F,968,0.840634
6,G,1040,0.709701
7,H,933,0.800453


,score,mean_prediction,count
0,0.00,0.101372,1522
1,0.25,0.291068,2465
2,0.50,0.478669,2475
3,0.75,0.641745,832
4,1.00,1.032294,265


,id,anchor,target,context,score,pred,abs_error
5836,6af4f2d9506545f2,punch face,face punch,B05,0.00,0.845918,0.845918
972,f297ea79eab86071,boom hydraulic cylinder,boom a hydraulic cylinder,E02,0.00,0.808941,0.808941
3794,20a45b67a785caf3,imaging axis,imaging of axis,G02,0.00,0.799531,0.799531
125,fbf3be656271239a,activating position,active region,G06,0.00,0.710440,0.710440
3790,c9f4b17fdba12548,imaging axis,image of axis,G02,0.00,0.693747,0.693747
1154,12c1f4b230252489,breech mechanism,breech loading naval gun,B29,0.75,0.062551,0.687449
6005,31677aa02d309ada,return structure,return to structure,D01,0.00,0.653880,0.653880
2776,463717d347b289d7,end pins,wood pin,G10,0.75,0.106178,0.643822
856,2b476e0af28051f7,belt shifting,belt shifter,D06,0.50,1.125209,0.625209
4147,62f86edbc7548252,invertebrate,cold blooded animal,A01,0.75,0.128955,0.621045


### Error-analysis interpretation prompts

Use the computed tables and examples to write your own evidence-based interpretation in the report:
- Which CPC sections have the lowest Pearson correlation? Check sample counts before treating a small section as definitively hardest.
- Do mean predictions move toward the middle for true scores 0 and 1? Relate any compression to the imbalance in training labels.
- Inspect the ten largest errors and group them into types (abbreviations, field-specific meaning, lexical overlap with different meaning, etc.). Do not claim a label is wrong without justification.

## 9. Task 5 â€” retrain selected configuration on all labeled data and write submission.csv

In [10]:
# The validation split selects the configuration; the final submission model is retrained on all labeled rows.
# This is the correct final-training step: do not use hidden test labels or leaderboard scores to select settings.
start=time.time()
full_ds=make_dataset(train_raw, True, context_mode=best["context_mode"])
test_ds=make_dataset(test_raw, False, context_mode=best["context_mode"])
final_model=AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, num_labels=1, dtype=torch.float32, local_files_only=True
)
final_args=compatible_training_arguments(
    output_dir="/kaggle/working/final_trainer",
    num_train_epochs=best["epochs"], learning_rate=best["lr"],
    per_device_train_batch_size=best["batch_size"],
    per_device_eval_batch_size=best["batch_size"]*2,
    warmup_ratio=0.1, weight_decay=0.01,
    eval_strategy="no", save_strategy="no", logging_steps=100,
    fp16=torch.cuda.is_available(), report_to="none", seed=SEED, data_seed=SEED,
    dataloader_num_workers=2,
)
final_trainer=Trainer(model=final_model,args=final_args,train_dataset=full_ds,
                      data_collator=DataCollatorWithPadding(tokenizer))
final_trainer.train()
test_predictions=final_trainer.predict(test_ds).predictions.reshape(-1)
submission=pd.DataFrame({"id":test_raw["id"],"score":np.clip(test_predictions,0,1)})
assert list(submission.columns)==["id","score"]
assert len(submission)==len(test_raw)
assert submission.id.equals(test_raw.id)
assert submission.score.notna().all() and submission.score.between(0,1).all()
submission.to_csv("/kaggle/working/submission.csv",index=False)
print("Saved /kaggle/working/submission.csv | rows:",len(submission),"| final training minutes:",round((time.time()-start)/60,1))
display(submission.head())

Map:   0%|          | 0/36473 [00:00<?, ? examples/s]

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/input/notebooks/charankailasa/patent-weights/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight   

Step,Training Loss
100,0.106148
200,0.078058
300,0.069074
400,0.064512
500,0.060415
600,0.054994
700,0.050074
800,0.049644
900,0.047936
1000,0.046809


Saved /kaggle/working/submission.csv | rows: 36 | final training minutes: 6.2


,id,score
0,4112d61851461f60,0.603304
1,09e418c93a776564,0.734309
2,36baf228038e314b,0.519423
3,1f37ead645e7f0c8,0.330645
4,71a5b6ad068d531f,0.000000


## 10. Task 5 â€” submission and leaderboard record

After the notebook finishes successfully:

1. Confirm `/kaggle/working/submission.csv` exists and has `id,score`.
2. Save/commit the notebook version and submit it to the competition.
3. Record the **actual** public and private leaderboard scores shown by Kaggle after evaluation. These cannot be inferred from the validation Pearson score.

Fill the values below only after Kaggle reports them.


In [11]:
# Optional manual record â€” replace None with the scores Kaggle actually reports.
leaderboard_record = pd.DataFrame([{
    "validation_pearson": best["pearson"],
    "public_leaderboard_score": None,
    "private_leaderboard_score": None,
    "submission_date": None,
    "notes": "Fill leaderboard fields after Kaggle evaluation; do not treat validation Pearson as leaderboard score."
}])
display(leaderboard_record)
leaderboard_record.to_csv("/kaggle/working/leaderboard_record_template.csv", index=False)


,validation_pearson,public_leaderboard_score,private_leaderboard_score,submission_date,notes
0,0.816755,None,None,None,Fill leaderboard fields after Kaggle evaluatio...


## 10. Submission checklist

- [ ] `submission.csv` exists in `/kaggle/working/` and has `id,score` columns.
- [ ] Notebook runs top-to-bottom with Internet **off** after the model checkpoint has been attached as a Kaggle input.
- [ ] Commit using **Save Version â†’ Save & Run All**.
- [ ] Open the committed version and choose **Submit to Competition** (or submit from the competition page if the option is available).
- [ ] Save a screenshot of the Submissions tab showing public and private scores. Do not invent these scores in the report.
- [ ] Report all submissions and compare leaderboard scores with validation Pearson.

Kaggle code competitions require offline execution and a file named `submission.csv`; see the official [competition page](https://www.kaggle.com/competitions/us-patent-phrase-to-phrase-matching).